<a href="https://colab.research.google.com/github/springboardmentor1234r/B13-AirFly-Insights-Internship/blob/Devika_S_Nair/milestone_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**AirFly Insights: Data Visualization and Analysis of Airline Operations**

***Milestone 1: Data Foundation and Cleaning***

Goal: Analyze airline delays / sales / traffic patterns


In [3]:
from google.colab import drive
import pandas as pd

# Mounting Google Drive to the default directory
drive.mount('/content/drive')

# accessing the file using its path
file_path = '/content/drive/MyDrive/flights_sample_3m.csv'
df = pd.read_csv(file_path)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,...,DIVERTED,CRS_ELAPSED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",...,0.0,186.0,176.0,153.0,1065.0,NaN,NaN,NaN,NaN,NaN
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",...,0.0,235.0,236.0,189.0,1399.0,NaN,NaN,NaN,NaN,NaN
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",...,0.0,118.0,112.0,87.0,680.0,NaN,NaN,NaN,NaN,NaN
3,2023-03-06,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2295,MSP,"Minneapolis, MN",SFO,"San Francisco, CA",...,0.0,260.0,285.0,249.0,1589.0,0.0,0.0,24.0,0.0,0.0
4,2020-02-23,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,407,MCO,"Orlando, FL",DFW,"Dallas/Fort Worth, TX",...,0.0,181.0,182.0,153.0,985.0,NaN,NaN,NaN,NaN,NaN


Returns the number of rows and columns

In [4]:
df.shape

(3000000, 32)

Shows column data types and non-null counts.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Data columns (total 32 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   FL_DATE                  object 
 1   AIRLINE                  object 
 2   AIRLINE_DOT              object 
 3   AIRLINE_CODE             object 
 4   DOT_CODE                 int64  
 5   FL_NUMBER                int64  
 6   ORIGIN                   object 
 7   ORIGIN_CITY              object 
 8   DEST                     object 
 9   DEST_CITY                object 
 10  CRS_DEP_TIME             int64  
 11  DEP_TIME                 float64
 12  DEP_DELAY                float64
 13  TAXI_OUT                 float64
 14  WHEELS_OFF               float64
 15  WHEELS_ON                float64
 16  TAXI_IN                  float64
 17  CRS_ARR_TIME             int64  
 18  ARR_TIME                 float64
 19  ARR_DELAY                float64
 20  CANCELLED                float64
 21  CANCELLA

counts the number of missing (null) values in the DEP_DELAY, ARR_DELAY, and CANCELLED columns.

In [6]:
df[['DEP_DELAY','ARR_DELAY','CANCELLED']].isnull().sum()

,0
DEP_DELAY,77644
ARR_DELAY,86198
CANCELLED,0


DEP_DELAY and ARR_DELAY to 0 for all rows where flights are marked as cancelled (CANCELLED == 1).

In [7]:
df.loc[df['CANCELLED'] == 1, ['DEP_DELAY','ARR_DELAY']] = 0

Replaces missing values in DEP_DELAY and ARR_DELAY with their respective median values for each column.

In [8]:
for col in ['DEP_DELAY','ARR_DELAY']:
    df[col].fillna(df[col].median(), inplace=True)

/tmp/ipykernel_11705/317644611.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)


Converts date/time columns, extracts month, weekday, and hour features, and creates a route column by combining origin and destination.

In [10]:
df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])

In [11]:
df['MONTH'] = df['FL_DATE'].dt.month

In [12]:
df['DAY_OF_WEEK'] = df['FL_DATE'].dt.dayofweek

In [13]:
df['DEP_TIME'] = pd.to_datetime(df['DEP_TIME'], format='%H%M', errors='coerce').dt.time

In [14]:
import numpy as np
df['DEP_HOUR'] = df['DEP_TIME'].apply(lambda x: x.hour if pd.notna(x) else np.nan)

In [15]:
df['ROUTE'] = df['ORIGIN'] + "-" + df['DEST']

In [16]:
print(df.shape)

df[['MONTH','DAY_OF_WEEK','DEP_HOUR','ROUTE']].head()

(3000000, 36)


,MONTH,DAY_OF_WEEK,DEP_HOUR,ROUTE
0,1,2,11.0,FLL-EWR
1,11,5,21.0,MSP-SEA
2,7,4,10.0,DEN-MSP
3,3,0,16.0,MSP-SFO
4,2,6,18.0,MCO-DFW


Saves the cleaned DataFrame to a Parquet file in Google Drive for efficient storage and later use.

In [17]:
df.to_parquet("/content/drive/MyDrive/airline_cleaned.parquet")